#### 首先根据决策因素确定因素矩阵,根据每个综合因素的pos与理想和负理想的欧几里得距离确定决策
#### 因为不一定因素越大就越好，故需要对矩阵先进行正向化(化为越大越好)
#### 极大型指标无需转换
#### 极小型指标 $1: max(v_i) - v_i \ \ 2: \frac{1}{v_i}$
#### 中间型指标 $\hat{x_i} = 1 - \frac{\left| x_{i} - x_{best} \right|}{max\{|x_i - x_{best}|\}}$
#### 区间型指标 假设最佳区间为$[a,b]$ 则
####          $M = \max\{a - \min\{x_i\},\max\{x_i\} - b\}$   
####         $
    \tilde{x_i} = \begin{cases}
              1 - \frac{a - x_i}{M},& x_i \lt a \\
              1,& a \leq x_i \leq b \\
              1 - \frac{x_i - b}{M},& x_i \gt b
                \end{cases}
              $

                                                                       

## 矩阵的输入处理

In [38]:
import numpy as np

n = int(input("参评数目n:>")) # 因素矩阵参数
m = int(input("指标个数m:>"))

kind = list(map(int,input("输入每列类型: 1.极大型 2.极小型 3.中间型 4.区间型\n").split()))

A = np.zeros((n,m))  # 开一个n x m 的0矩阵
print(f"请输入{n}x{m}的矩阵")
for i in range(n):
    A[i] = list(map(float,input().split()))

print(f"矩阵为:\n{A}")


请输入3x4的矩阵
矩阵为:
[[  9.  10. 175. 120.]
 [  8.   7. 164.  80.]
 [  6.   3. 157.  90.]]


## 矩阵的正向化处理函数

In [46]:
def min_to_max(x : np.ndarray):
    return x.max() - x # 直接对数值处理

def min_to_max2(x): # 极小化极大
    x = list(x) # 先化为列表
    maxv = max(x)
    ret = [[maxv - e] for e in x] # 使用列表推导式语法 生成一个列向量列表
    return np.array(ret)  # 返回二维的列向量ndarray

def mid_to_max(x : np.ndarray,bestv): # 暂时不写
    return
def mid_to_max2(x,bestv): # 中间化极大
    x = list(x)
    h = [abs(val - bestv) for val in x] # 先计算分母，找到距离的最大值
    M = max(h)
    if(M == 0): # 这里预期的M是一个float,所以后续可能要改进
        M = 1 # 若M为0 则可任意指定一个值兼容后续的计算，防止分母为0
    ret = [[1 - (val / M)] for val in h]
    return np.array(ret)

def reg_to_max2(x,l,r): # 区间化极大
    x = list(x)
    M = max(l - min(x),max(x) - r) # 暂时不考虑特判0,因为如果M=0 理论计算不会用到M
    ret = []
    for v in x:
        if v < l:
            ret.append([1 - (l - v) / M])
        elif v > r:
            ret.append([1 - (v - r) / M])
        else:
            ret.append([1])
    return np.array(ret)




## 矩阵的正向化处理

In [48]:
fA = np.zeros((n,1))
for i in range(m):
    match kind[i]:
        case 1:
            ans = A[:,i].reshape(n,1)
        case 2:
            ans = min_to_max2(A[:,i])
        case 3:
            val = float(input("请输入最优值"))
            ans = mid_to_max2(A[:,i],val)
        case 4:
            l,r = map(float,input("请输入最优区间[a,b]").split())
            ans = reg_to_max2(A[:,i],l,r)
        case _:
            raise ValueError(f"The value of kind {kind[i]} should be 1 to 4")
    if i == 0:
        fA = ans
    else:
        fA = np.hstack((fA,ans))

print(f"一致矩阵正向化后为\n{fA}")

一致矩阵正向化后为
[[9.  0.  0.  0. ]
 [8.  3.  0.9 0.5]
 [6.  7.  0.2 1. ]]


## 矩阵的标准化处理

In [50]:
for i in range(m):
    fA[:,i] = fA[:,i] / np.sqrt(sum(fA[:,i]**2))
print(f"标准化后的矩阵为\n{fA}")

标准化后的矩阵为
[[0.66896473 0.         0.         0.        ]
 [0.59463532 0.3939193  0.97618706 0.4472136 ]
 [0.44597649 0.91914503 0.21693046 0.89442719]]


## 按距离决定最终得分
### (以下为假设权重相同的情况，若要定义权重需要结合其他决策方法如层次分析法)

In [51]:
maxv = np.max(fA,axis = 0) # 按第一维的几何方向压缩最大值 即计算每一列的最大值返回
minv = np.min(fA,axis = 0)

print("理想点: ",maxv)
print("负理想点: ",minv)

d1 = np.sqrt(np.sum((fA - maxv)**2,axis = 1)) # 计算每一个对象与理想点的距离
d2 = np.sqrt(np.sum((fA - minv)**2,axis = 1))

print("d+: ",d1)
print("d-: ",d2)

s = d2 / (d1 + d2)  # 计算每个对象的得分
score = (s * 100) / sum(s) # 化为百分制

print("原始得分结果: ",s)
print(f"百分比得分结果: {score}")


理想点:  [0.66896473 0.91914503 0.97618706 0.89442719]
负理想点:  [0.44597649 0.         0.         0.        ]
d+:  [1.61175952 0.69382053 0.79132442]
d-:  [0.22298824 1.15334862 1.30072534]
原始得分结果:  [0.12153618 0.62438712 0.62174684]
百分比得分结果: [ 8.88636674 45.65334106 45.46029221]
